# Quality check: is this corpus any good?

A bird's-eye view of the series layer, meant to be run after a crawl and re-run as it grows.
It answers the questions you need answered *before* choosing what to train on:

- how many series are there, and how much data does each really carry?
- how many are still being updated, and how many reach a given cutoff?
- how many have half a seasonal cycle, one, two, three?
- how much is missing, and are the holes in the middle or at the edges?
- how many are constant, or so rounded they barely move?

Every number comes from `terrastat.quality`, so the notebook and the batch script cannot drift
apart. To get the same tables written to disk without opening a notebook:

```powershell
.\.venv\Scripts\python -m terrastat.quality --out reports\quality
```

In [1]:
import datetime as dt
import polars as pl
from terrastat import quality

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(30)

ASOF  = dt.date.today()          # reference point for "how stale is this series"
UNTIL = dt.date(2025, 12, 1)     # the cutoff you care about reaching
# Scope. FRED by default so this notebook runs anywhere: the scalar passes are cheap over the
# whole corpus, but the sampled value-level pass and the exact quantiles are not, and running
# them over all 956M series inside a notebook kernel will exhaust memory on most machines.
# Set SOURCES = None for everything, preferably from the command line:
#     terrastat quality --out reports/quality
SOURCES = ["fred"]               # None = all three
FREQ    = None                   # e.g. ["M", "Q"]; None = all

ASOF, UNTIL

(datetime.date(2026, 9, 8), datetime.date(2025, 12, 1))

## 1. How much is there

The scalar pass reads only the fixed-width columns, never the `dates`/`values`/`flags` lists that
make up almost all of the 28 GB on disk. That is why it covers the whole population exactly and
still returns in seconds.

`obs_p50` is the median number of observations per series, and it is the number to look at
first: a source can contribute hundreds of millions of *series* while contributing very little
*data*, because SDMX dimension cross-products multiply thin slices. The quartiles either side of
it are exact, computed from a count of each distinct length rather than by sorting a billion-row
column (which does not fit in memory).

In [2]:
ov = quality.overview(sources=SOURCES, frequencies=FREQ, asof=ASOF, until=UNTIL)
ov.select("source", "frequency", "n_series", "n_obs",
          "obs_p25", "obs_p50", "obs_p75",
          pl.col("years_median").round(1).alias("yrs_med"),
          "earliest", "latest")

source,frequency,n_series,n_obs,obs_p25,obs_p50,obs_p75,yrs_med,earliest,latest
str,str,u32,i64,i64,i64,i64,f64,date,date
"""fred""","""A""",493263,14429828,16,27,36,27.0,1086-01-01,2031-01-01
"""fred""","""BW""",16,12108,504,905,936,34.8,1954-07-14,2026-09-03
"""fred""","""D""",11478,49589982,1841,3896,6524,10.7,1854-12-01,2026-09-04
"""fred""","""M""",224015,59856984,109,212,438,17.7,1694-11-01,2027-09-01
"""fred""","""OTHER""",62,5692,70,85,98,null,1914-11-16,2026-08-09
"""fred""","""P""",842,9289,10,12,12,60.0,1910-01-01,2017-01-01
"""fred""","""Q""",110098,17119234,85,123,251,30.8,1695-01-01,2036-10-01
"""fred""","""S""",2113,135266,48,68,85,34.0,1863-07-01,2026-01-01
"""fred""","""W""",3631,4346984,863,1073,1238,20.6,1855-01-05,2026-09-03


In [3]:
# the same thing as a share of the corpus, which is usually the more sobering view
tot_s, tot_o = ov["n_series"].sum(), ov["n_obs"].sum()
(ov.select("source", "frequency",
           (pl.col("n_series") / tot_s * 100).round(2).alias("% of series"),
           (pl.col("n_obs") / tot_o * 100).round(2).alias("% of observations"))
   .sort("% of series", descending=True)
   .head(12))

source,frequency,% of series,% of observations
str,str,f64,f64
"""fred""","""A""",58.34,9.92
"""fred""","""M""",26.49,41.14
"""fred""","""Q""",13.02,11.77
"""fred""","""D""",1.36,34.08
"""fred""","""W""",0.43,2.99
"""fred""","""S""",0.25,0.09
"""fred""","""P""",0.1,0.01
"""fred""","""OTHER""",0.01,0.0
"""fred""","""BW""",0.0,0.01


## 2. Length, in cycles

A cycle is one year: 12 points at monthly, 4 at quarterly, 1 at annual. Length is measured in
years of *actual data* (`n_obs / periods_per_year`), not calendar span, because a series with a
ten-year span and four observations cannot be modelled as if it had ten years.

Half a cycle is the bare minimum for anything seasonal to be visible at all; two or three cycles
is where seasonal estimation starts to behave.

In [4]:
ln = quality.length_table(sources=SOURCES, frequencies=FREQ)
ln

source,frequency,n_series,ge_0.5_cycles,ge_1_cycles,ge_2_cycles,ge_3_cycles,ge_5_cycles,ge_10_cycles,ge_20_cycles
str,str,u32,u32,u32,u32,u32,u32,u32,u32
"""fred""","""A""",493263,493263,493263,490682,489051,485121,471030,311210
"""fred""","""BW""",16,16,16,16,16,14,14,9
"""fred""","""D""",11478,11469,11416,11081,10080,8685,6291,297
"""fred""","""M""",224015,222832,222293,221360,220635,217997,145554,105920
"""fred""","""OTHER""",62,0,0,0,0,0,0,0
"""fred""","""P""",842,842,842,842,842,842,842,836
"""fred""","""Q""",110098,109847,109567,109116,108855,108314,103381,89600
"""fred""","""S""",2113,2113,2113,2113,2113,2113,1930,1675
"""fred""","""W""",3631,3624,3620,3583,3409,3352,3036,1979


In [5]:
# as percentages, which is easier to read across sources of very different size
cuts = [c for c in ln.columns if c.startswith("ge_")]
ln.select("source", "frequency", "n_series",
          *[(pl.col(c) / pl.col("n_series") * 100).round(1).alias(c.replace("ge_", "%>=")) for c in cuts])

source,frequency,n_series,%>=0.5_cycles,%>=1_cycles,%>=2_cycles,%>=3_cycles,%>=5_cycles,%>=10_cycles,%>=20_cycles
str,str,u32,f64,f64,f64,f64,f64,f64,f64
"""fred""","""A""",493263,100.0,100.0,99.5,99.1,98.3,95.5,63.1
"""fred""","""BW""",16,100.0,100.0,100.0,100.0,87.5,87.5,56.2
"""fred""","""D""",11478,99.9,99.5,96.5,87.8,75.7,54.8,2.6
"""fred""","""M""",224015,99.5,99.2,98.8,98.5,97.3,65.0,47.3
"""fred""","""OTHER""",62,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""fred""","""P""",842,100.0,100.0,100.0,100.0,100.0,100.0,99.3
"""fred""","""Q""",110098,99.8,99.5,99.1,98.9,98.4,93.9,81.4
"""fred""","""S""",2113,100.0,100.0,100.0,100.0,100.0,91.3,79.3
"""fred""","""W""",3631,99.8,99.7,98.7,93.9,92.3,83.6,54.5


## 3. Recency

`fresh_le_Np` is the share whose last observation is within N periods of `ASOF`, in the series'
own frequency: for monthly, `fresh_le_2p` is the "updated within two months" figure.

Two things to keep in mind. The lag is measured from `ASOF`, so it includes however long ago the
crawl ran — set `ASOF` to the crawl date if you want publication lag alone. And a series can be
fresh and still useless if it is two points long, which is why the task table below combines
recency with length rather than reporting it on its own.

In [6]:
fresh = [c for c in ov.columns if c.startswith("fresh_")]
reaches = [c for c in ov.columns if c.startswith("reaches_")]
ov.select("source", "frequency",
          *[(pl.col(c) * 100).round(1).alias(c) for c in fresh + reaches])

source,frequency,fresh_le_1p,fresh_le_2p,fresh_le_3p,fresh_le_6p,fresh_le_12p,reaches_2025_12
str,str,f64,f64,f64,f64,f64,f64
"""fred""","""A""",0.3,22.1,73.3,83.4,93.7,0.3
"""fred""","""BW""",12.5,12.5,12.5,12.5,12.5,12.5
"""fred""","""D""",0.0,0.0,0.0,87.0,90.4,91.4
"""fred""","""M""",0.0,0.6,73.4,78.2,80.0,79.9
"""fred""","""OTHER""",null,null,null,null,null,90.3
"""fred""","""P""",0.0,26.6,80.3,99.9,100.0,0.0
"""fred""","""Q""",0.5,15.3,60.8,65.6,70.7,60.8
"""fred""","""S""",0.0,65.3,65.4,69.9,70.2,65.3
"""fred""","""W""",20.9,24.5,37.5,54.5,55.9,56.1


## 4. Completeness

`missing_share` is `1 - n_obs / n_points`: the fraction of stored periods with no value. Sources
differ structurally here — FRED stores only the points it has, so its series are complete by
construction, while the SDMX sources lay out a full rectangle and leave holes.

In [7]:
comp = ["missing_share_mean", "share_complete", "flagged_share_mean"]
ov.select("source", "frequency", *[(pl.col(c) * 100).round(1).alias(c) for c in comp])

source,frequency,missing_share_mean,share_complete,flagged_share_mean
str,str,f64,f64,f64
"""fred""","""A""",0.0,100.0,0.0
"""fred""","""BW""",0.0,100.0,0.0
"""fred""","""D""",0.0,100.0,0.0
"""fred""","""M""",0.0,100.0,0.0
"""fred""","""OTHER""",0.0,100.0,0.0
"""fred""","""P""",0.0,100.0,0.0
"""fred""","""Q""",0.0,100.0,0.0
"""fred""","""S""",0.0,100.0,0.0
"""fred""","""W""",0.0,100.0,0.0


## 5. What is each slice actually good for (univariate-local)

These profiles are simple predicates, defined in one place in `quality.profiles()` so they can be
argued with. They are a starting point, not a standard.

They are **univariate-local** requirements: they ask whether one series, alone, carries enough
history to identify a model fitted to it alone. For a global model cross-learning across the
corpus that is the wrong test — see the aligned cohort below.

| profile | rule |
|---|---|
| `trainable` | >= 2 horizons, <= 50% missing |
| `real_time` | last point within 2 periods, >= 3 horizons, <= 20% missing |
| `nowcasting` | within 3 periods, >= 4 horizons, <= 20% missing |
| `backtest` | >= 3 horizons, <= 20% missing (no recency requirement) |
| `seasonal` | >= 3 cycles, at least quarterly, <= 20% missing |
| `imputation` | 1-50% missing, >= 2 horizons (a complete series is no use here) |
| `long_history` | spans >= 20 calendar years |

Length is measured in **forecast horizons**, following the M4 conventions (annual 6, quarterly 8,
monthly 18, weekly 13, daily 14). Neither obvious alternative works: measuring in years makes one
annual point a whole year of data, and a flat observation count scales the calendar requirement
inversely with frequency, so "60 observations" quietly demands sixty years of an annual series
and five of a monthly one. A horizon means the same thing everywhere — a backtest needs enough
history to hold one out and still fit on what is left.

In observations that comes to:

| freq | horizon | trainable | backtest | nowcasting |
|---|---|---|---|---|
| A | 6 | 12 | 18 | 24 |
| Q | 8 | 16 | 24 | 32 |
| M | 18 | 36 | 54 | 72 |
| W | 13 | 26 | 39 | 52 |
| D | 14 | 28 | 42 | 56 |

In [8]:
tk = quality.task_table(sources=SOURCES, frequencies=FREQ, asof=ASOF, until=UNTIL)
tk

source,frequency,n_series,trainable,real_time,nowcasting,backtest,seasonal,imputation,long_history,reaches_cutoff
str,str,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""fred""","""A""",493263,463799,107382,228647,326248,0,0,319111,1550
"""fred""","""BW""",16,16,2,2,16,16,0,9,2
"""fred""","""D""",11478,11471,0,0,11471,10080,0,4811,10491
"""fred""","""M""",224015,220635,1322,163455,218303,220635,0,107477,179045
"""fred""","""OTHER""",62,0,0,0,0,0,0,6,56
"""fred""","""P""",842,809,224,0,440,0,0,834,0
"""fred""","""Q""",110098,108520,16728,66586,107648,108855,0,90898,66892
"""fred""","""S""",2113,2113,1380,1382,2113,0,0,1672,1380
"""fred""","""W""",3631,3624,891,1361,3624,3409,0,1980,2037


In [9]:
# the totals, which is the line to quote in a paper
totals = tk.select(pl.exclude("source", "frequency").sum())
n = totals["n_series"][0]
pl.DataFrame({
    "profile": [c for c in totals.columns if c != "n_series"],
    "series": [totals[c][0] for c in totals.columns if c != "n_series"],
}).with_columns((pl.col("series") / n * 100).round(2).alias("% of corpus"))

profile,series,% of corpus
str,i64,f64
"""trainable""",810987,95.92
"""real_time""",127929,15.13
"""nowcasting""",461433,54.57
"""backtest""",669863,79.23
"""seasonal""",342995,40.57
"""imputation""",0,0.0
"""long_history""",526798,62.3
"""reaches_cutoff""",261453,30.92


## 6. Value-level checks (sampled)

These need the `values` column, so they read the bulk of the data and run on a stratified sample.
Raise `sample` if you want tighter estimates and can wait.

`unique_share` is distinct values divided by observations. It is the cheapest detector of series
that inflate a corpus without teaching anything: a constant sits at ~0, a step function or a
heavily rounded index sits very low, a genuine economic series sits high.

`share_interior_gap` is the one that matters for imputation — a hole between two observations is
a problem to solve, whereas missing points at the start or end just mean a shorter series.

In [10]:
dp = quality.deep_table(sources=SOURCES, frequencies=FREQ, sample=30_000)
dp.select("source", "frequency", "sampled",
          *[(pl.col(c) * 100).round(1).alias(c) for c in dp.columns
            if c not in ("source", "frequency", "sampled")])

source,frequency,sampled,datasets,share_one_point,n_multi,unique_share_mean,unique_share_median,share_constant,share_near_constant,share_any_gap,share_interior_gap
str,str,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64
"""fred""","""A""",11883,10400,0.4,1183200,91.3,100.0,1.6,1.0,0.0,0.0
"""fred""","""BW""",16,400,0.0,1600,75.3,99.2,0.0,12.5,0.0,0.0
"""fred""","""D""",1141,4700,0.0,114100,42.0,33.9,0.0,20.6,0.0,0.0
"""fred""","""M""",9020,15200,0.5,897300,72.0,84.7,0.8,2.9,0.0,0.0
"""fred""","""OTHER""",62,800,0.0,6200,36.8,36.4,3.2,11.3,0.0,0.0
"""fred""","""P""",314,300,0.0,31400,99.2,100.0,0.0,0.0,0.0,0.0
"""fred""","""Q""",6457,7200,0.0,645600,82.8,99.0,0.8,1.6,0.0,0.0
"""fred""","""S""",301,300,0.0,30100,98.3,100.0,0.0,0.0,0.0,0.0
"""fred""","""W""",1419,3300,0.1,141700,57.3,64.0,3.1,10.4,0.0,0.0


## 7. Zoom in: monthly

The original question, spelled out for one frequency.

In [11]:
FOCUS = "M"
lf = quality._with_derived(quality.scan(columns=quality.SCALAR_COLUMNS, frequencies=[FOCUS]), ASOF)
m = lf.collect(engine="streaming")

print(f"{FOCUS}: {m.height:,} series, {m['n_obs'].sum():,} observations")
for label, expr in [
    ("updated within 2 months",      pl.col("lag_periods") <= 2),
    (f"reaching {UNTIL:%Y-%m}",      pl.col("end_date") >= UNTIL),
    ("at least half a cycle (6m)",   pl.col("years") >= 0.5),
    ("at least 1 cycle (12m)",       pl.col("years") >= 1),
    ("at least 2 cycles (24m)",      pl.col("years") >= 2),
    ("at least 3 cycles (36m)",      pl.col("years") >= 3),
    ("no missing values",            pl.col("missing_share") == 0),
]:
    k = m.select(expr.sum()).item()
    print(f"  {label:28} {k:>12,}  ({k / m.height:6.2%})")

M: 9,688,979 series, 991,632,661 observations
  updated within 2 months            54,711  ( 0.56%)
  reaching 2025-12                7,785,666  (80.36%)
  at least half a cycle (6m)      9,286,616  (95.85%)
  at least 1 cycle (12m)          9,129,034  (94.22%)
  at least 2 cycles (24m)         8,642,215  (89.20%)
  at least 3 cycles (36m)         7,511,900  (77.53%)
  no missing values               9,426,409  (97.29%)


In [12]:
# where the mass actually sits: series counted by how many years of data they carry
(m.with_columns(pl.col("years").cut([0.5, 1, 2, 3, 5, 10, 20]).alias("bucket"))
   .group_by("bucket").agg(pl.len().alias("series"), pl.col("n_obs").sum().alias("obs"))
   .sort("bucket"))

bucket,series,obs
enum,u32,i64
"""(-inf, 0.5]""",437196,1113503
"""(0.5, 1]""",266867,2784655
"""(1, 2]""",599401,11988630
"""(2, 3]""",1043946,32534501
"""(3, 5]""",1490632,73088939
"""(5, 10]""",2881121,212837911
"""(10, 20]""",2173518,395052045
"""(20, inf]""",796298,262232477


## 6b. The aligned cohort

Everything above judges a series **on its own**: can this one series, by itself, identify a model
fitted to it alone? That is the right question for local methods and the wrong one for a global
model that cross-learns, where a short series is not unidentifiable because its parameters come
from the corpus rather than from itself.

So this is a different cut. Fix a forecast time — say the end of 2025, to forecast 2026 — and keep
every series that still has data at that point and already had data a year earlier, whatever its
frequency. They can then be lined up on a common calendar. Roll the origin back a year at a time
and the same cohort gives you rolling-origin cross-validation: forecast 2025 from the end of 2024,
2024 from the end of 2023, and so on.

The length bar here is one year, deliberately, not the horizon multiples above.

One thing the scalar columns cannot tell us: the endpoints prove the series *brackets* the window
and `max_missing` bounds the holes overall, but neither proves the window itself is unbroken.
Checking that needs the dates list.

In [13]:
ORIGIN = quality._default_origin()      # the most recent 31 December
al = quality.aligned_table(sources=SOURCES, frequencies=FREQ, origin=ORIGIN, folds=3)
print(f"forecast time {ORIGIN}; folds_k = usable with the origin rolled back k years")
al

forecast time 2025-12-31; folds_k = usable with the origin rolled back k years


source,frequency,n_series,folds_0,folds_1,folds_2,folds_3
str,str,u32,u32,u32,u32,u32
"""fred""","""A""",493263,108871,108871,108855,108832
"""fred""","""BW""",16,2,2,2,2
"""fred""","""D""",11478,10314,10109,9572,9231
"""fred""","""M""",224015,178756,178512,177428,177063
"""fred""","""OTHER""",62,0,0,0,0
"""fred""","""P""",842,0,0,0,0
"""fred""","""Q""",110098,68926,68782,68777,68772
"""fred""","""S""",2113,1380,1380,1380,1380
"""fred""","""W""",3631,2037,2037,2037,2037


In [14]:
# how the cohort decays as the origin rolls back, as a share of each slice
al.select("source", "frequency", "n_series",
          *[(pl.col(c) / pl.col("n_series") * 100).round(1).alias(c) for c in al.columns
            if c.startswith("folds_")])

source,frequency,n_series,folds_0,folds_1,folds_2,folds_3
str,str,u32,f64,f64,f64,f64
"""fred""","""A""",493263,22.1,22.1,22.1,22.1
"""fred""","""BW""",16,12.5,12.5,12.5,12.5
"""fred""","""D""",11478,89.9,88.1,83.4,80.4
"""fred""","""M""",224015,79.8,79.7,79.2,79.0
"""fred""","""OTHER""",62,0.0,0.0,0.0,0.0
"""fred""","""P""",842,0.0,0.0,0.0,0.0
"""fred""","""Q""",110098,62.6,62.5,62.5,62.5
"""fred""","""S""",2113,65.3,65.3,65.3,65.3
"""fred""","""W""",3631,56.1,56.1,56.1,56.1


In [15]:
# the cohort itself, ready to feed a training set or an export
cohort = quality.aligned_series(sources=SOURCES, frequencies=FREQ, origin=ORIGIN, folds=1)
print(f"{cohort.select(pl.len()).collect().item():,} series aligned with one fold of rollback")
cohort.select("series_uid", "source", "frequency", "start_date", "end_date", "n_obs").head(8).collect()

369,693 series aligned with one fold of rollback


series_uid,source,frequency,start_date,end_date,n_obs
str,str,str,date,date,i64
"""fred:SBCSS10B0V0FRB""","""fred""","""A""",2017-01-01,2025-01-01,9
"""fred:SBCSS10B10V0FRB""","""fred""","""A""",2017-01-01,2025-01-01,9
"""fred:SBCSS10B10V1FRB""","""fred""","""A""",2017-01-01,2025-01-01,9
"""fred:SBCSS10B11V0FRB""","""fred""","""A""",2017-01-01,2025-01-01,9
"""fred:SBCSS10B11V1FRB""","""fred""","""A""",2017-01-01,2025-01-01,9
"""fred:SBCSS10B1V1FRB""","""fred""","""A""",2017-01-01,2025-01-01,9
"""fred:SBCSS10B1V2FRB""","""fred""","""A""",2017-01-01,2025-01-01,9
"""fred:SBCSS10B1V3FRB""","""fred""","""A""",2017-01-01,2025-01-01,9


## 7b. Down one level: per dataset

Everything above is per **source** (FRED / Eurostat / OECD) and frequency. The same questions one
level down are what you actually sort when choosing what to train on, because a source-level
number hides enormous variation: one Eurostat dataset can contribute a hundred million
two-observation series while the one beside it contributes ten thousand usable ones.

This reads `dataset_title`, a string column, which makes it the slowest pass here — hence
`--datasets` rather than on by default in the batch script.

In [16]:
ds = quality.dataset_table(sources=SOURCES, frequencies=FREQ, asof=ASOF, until=UNTIL)
print(f"{ds.height:,} dataset x frequency combinations")
ds.head(15).select("source", "dataset_id", "frequency",
                   pl.col("title").str.slice(0, 40).alias("title"),
                   "n_series", "obs_mean", "backtest", "real_time", "latest")

426 dataset x frequency combinations


source,dataset_id,frequency,title,n_series,obs_mean,backtest,real_time,latest
str,str,str,str,u32,f64,u32,u32,date
"""fred""","""346""","""A""","""Small Area Income and Poverty …",80956,30.4,80589,0,2024-01-01
"""fred""","""429""","""A""","""County Population Estimates By…",65971,16.0,0,0,2024-01-01
"""fred""","""462""","""M""","""Housing Inventory Core Metrics""",64459,110.0,63271,0,2026-07-01
"""fred""","""308""","""M""","""State and Metro Area Employmen…",39655,371.8,39655,0,2026-07-01
"""fred""","""205""","""Q""","""Main Economic Indicators""",36075,124.3,36012,4107,2026-04-01
"""fred""","""397""","""A""","""Gross Domestic Product by Coun…",26694,22.4,23991,0,2024-01-01
"""fred""","""205""","""A""","""Main Economic Indicators""",26580,31.8,23810,13452,2025-01-01
"""fred""","""463""","""M""","""Market Hotness Index""",26000,104.9,25920,0,2026-07-01
"""fred""","""52""","""A""","""Z.1 Financial Accounts of the …",23432,76.5,23164,19648,2025-01-01


In [17]:
# the datasets that actually carry usable data, rather than the ones with the most rows
(ds.filter(pl.col("backtest") > 0)
   .sort("backtest", descending=True)
   .head(15)
   .select("source", "dataset_id", "frequency",
           pl.col("title").str.slice(0, 40).alias("title"),
           "n_series", "backtest",
           (pl.col("backtest") / pl.col("n_series") * 100).round(1).alias("% usable")))

source,dataset_id,frequency,title,n_series,backtest,% usable
str,str,str,str,u32,u32,f64
"""fred""","""346""","""A""","""Small Area Income and Poverty …",80956,80589,99.5
"""fred""","""462""","""M""","""Housing Inventory Core Metrics""",64459,63271,98.2
"""fred""","""308""","""M""","""State and Metro Area Employmen…",39655,39655,100.0
"""fred""","""205""","""Q""","""Main Economic Indicators""",36075,36012,99.8
"""fred""","""463""","""M""","""Market Hotness Index""",26000,25920,99.7
"""fred""","""397""","""A""","""Gross Domestic Product by Coun…",26694,23991,89.9
"""fred""","""205""","""A""","""Main Economic Indicators""",26580,23810,89.6
"""fred""","""52""","""A""","""Z.1 Financial Accounts of the …",23432,23164,98.9
"""fred""","""52""","""Q""","""Z.1 Financial Accounts of the …",23043,22861,99.2


## 8. Dates worth a second look

None of these is necessarily an error. Dates past today are usually projections — FRED carries
CBO and OMB forecast series that legitimately run into the 2030s — and a start in the 1600s can
be a real historical reconstruction. They are listed because a mis-parsed period looks exactly
the same from here, and because a forecast series left in a training set is a leak.

In [18]:
quality.anomalies(sources=SOURCES, frequencies=FREQ, asof=ASOF)

source,frequency,n_series,ends_in_future,starts_before_1800,single_point,empty,end_before_start
str,str,u32,u32,u32,u32,u32,u32
"""fred""","""A""",493263,543,46,2581,0,0
"""fred""","""BW""",16,0,0,0,0,0
"""fred""","""D""",11478,0,0,0,0,0
"""fred""","""M""",224015,8,7,364,0,0
"""fred""","""OTHER""",62,0,0,0,0,0
"""fred""","""P""",842,0,0,0,0,0
"""fred""","""Q""",110098,4,6,251,0,0
"""fred""","""S""",2113,0,0,0,0,0
"""fred""","""W""",3631,0,0,2,0,0


In [19]:
# what the future-dated series actually are, so you can judge them
(quality.scan(columns=["series_uid", "title", "frequency", "start_date", "end_date"])
   .filter(pl.col("end_date") > ASOF)
   .head(10)
   .collect())

series_uid,title,frequency,start_date,end_date
str,str,str,date,date
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01
"""eurostat:met_proj_19ranmig:A.B…","""Assumptions for net migration …","""A""",2019-01-01,2100-01-01


## Running it in batch

The same tables, written to disk as Parquet and CSV plus a short written summary, with no
notebook involved:

```powershell
.\.venv\Scripts\python -m terrastat.quality --out reports\quality
.\.venv\Scripts\python -m terrastat.quality --sources eurostat --freq M Q --no-deep
.\.venv\Scripts\python -m terrastat.quality --asof 2026-09-04 --until 2025-12-01 --sample 200000
```

`--asof` is worth setting to the date the crawl finished: recency is measured from it, so leaving
it at today counts the age of the crawl as staleness in the data.